# LSTM Experiment E Historical Traffic Weather

**24-hour input lookback → next-hour EV charging demand.**

Extracted from `Untitled0.ipynb`. Original execution outputs are retained. Run cells in order in Google Colab; confirm Google Drive paths before executing.

## Data loading, preprocessing, model training, evaluation and saved results

## Notebook Setup

In [ ]:

# ============================================================
# EXPERIMENT K
# LSTM - HISTORICAL DEMAND + TRAFFIC + WEATHER
# 24-HOUR LOOKBACK -> NEXT-HOUR DEMAND
# ============================================================

# ============================================================


## 1. Import Libraries and Mount Google Drive

In [ ]:
# 1. IMPORTS AND GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)


# ============================================================


## 2. Experiment Configuration

In [ ]:
# 2. CONFIGURATION
# ============================================================

ROOT = "/content/drive/MyDrive/EV_Load_Forecasting_FYP"

DATASET_PATH = os.path.join(
    ROOT,
    "Processed Data",
    "Hourly",
    "experiment_c_final_2019.csv"
)

EXP_DIR = os.path.join(
    ROOT,
    "Experiments",
    "LSTM_Experiment_K_Historical_Traffic_Weather_2019"
)

os.makedirs(EXP_DIR, exist_ok=True)

MODEL_PATH = os.path.join(
    EXP_DIR,
    "lstm_experiment_k_best.keras"
)

LOOKBACK = 24
BATCH_SIZE = 32
EPOCHS = 200
LEARNING_RATE = 0.001

WEATHER_COLS = [
    "clouds_lag1d",
    "dewpt_lag1d",
    "ghi_lag1d",
    "max_uv_lag1d",
    "precip_lag1d",
    "rh_lag1d",
    "slp_lag1d",
    "temp_lag1d",
    "wind_gust_spd_lag1d",
    "wind_spd_lag1d"
]

RAW_FEATURE_COLS = [
    "hourly_energy_kwh",
    "total_traffic"
] + WEATHER_COLS

print("Experiment directory:", EXP_DIR)


# ============================================================


## 3. Load and Inspect Dataset

In [ ]:
# 3. LOAD AND VERIFY DATASET
# ============================================================

df = pd.read_csv(
    DATASET_PATH,
    parse_dates=["date_hour"]
)

df = (
    df.sort_values("date_hour")
    .reset_index(drop=True)
)

required_cols = ["date_hour"] + RAW_FEATURE_COLS

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing dataset columns: {missing_cols}"
    )

df = df[required_cols].copy()

for col in RAW_FEATURE_COLS:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print("\n" + "=" * 70)
print("EXPERIMENT K - DATASET INFORMATION")
print("=" * 70)

print("Dataset shape:", df.shape)
print("Date range:",
      df["date_hour"].min(),
      "to",
      df["date_hour"].max())

print("\nMissing values:")
print(df.isna().sum())

print("\nFirst five rows:")
print(df.head())

assert len(df) == 8016
assert df["hourly_energy_kwh"].isna().sum() == 206
assert not df["date_hour"].duplicated().any()

expected_dates = pd.date_range(
    df["date_hour"].min(),
    df["date_hour"].max(),
    freq="h"
)

assert np.array_equal(
    expected_dates.to_numpy(),
    df["date_hour"].to_numpy()
), "Dataset contains missing or irregular hourly timestamps."


# ============================================================


## 4. Chronological Data Splitting

In [ ]:
# 4. CHRONOLOGICAL SPLIT
# ============================================================

n_total = len(df)

train_end = int(n_total * 0.70)
val_end = int(n_total * 0.85)

assert train_end == 5611
assert val_end == 6813

print("\n" + "=" * 70)
print("CHRONOLOGICAL SPLITTING")
print("=" * 70)

for name, part in [
    ("Training", df.iloc[:train_end]),
    ("Validation", df.iloc[train_end:val_end]),
    ("Testing", df.iloc[val_end:])
]:
    print(
        f"{name}: {len(part)} rows | "
        f"{part['date_hour'].min()} to "
        f"{part['date_hour'].max()}"
    )

    print(
        "Missing demand:",
        part["hourly_energy_kwh"].isna().sum()
    )


# ============================================================


## 5. Missing Value Handling

In [ ]:
# 5. MISSING VALUE HANDLING
# ============================================================

processed = df.copy()

# Preserve observed demand as target.
# Missing targets will be excluded from training/evaluation.
processed["target_original"] = (
    processed["hourly_energy_kwh"]
)

# Historical demand: causal forward-fill
processed["demand_input"] = (
    processed["hourly_energy_kwh"].ffill()
)

# Historical traffic: causal forward-fill
processed["traffic_input"] = (
    processed["total_traffic"].ffill()
)

# Weather columns are already lagged by one day.
# Forward-fill remaining missing observations.
processed[WEATHER_COLS] = (
    processed[WEATHER_COLS].ffill()
)

# Fill leading missing external values using training-only
# medians, if required.
external_cols = ["traffic_input"] + WEATHER_COLS

for col in external_cols:

    if processed[col].isna().any():

        train_median = (
            processed.iloc[:train_end][col].median()
        )

        if pd.isna(train_median):
            raise ValueError(
                f"No valid training observations for {col}"
            )

        processed[col] = (
            processed[col].fillna(train_median)
        )

if processed["demand_input"].isna().any():
    raise ValueError(
        "Historical demand has leading missing values. "
        "Check the dataset before continuing."
    )

FEATURE_COLS = [
    "demand_input",
    "traffic_input"
] + WEATHER_COLS

N_FEATURES = len(FEATURE_COLS)

print("\n" + "=" * 70)
print("MISSING VALUE HANDLING")
print("=" * 70)

print(processed[
    ["target_original"] + FEATURE_COLS
].isna().sum())

print("\nInput features:", FEATURE_COLS)
print("Number of input features:", N_FEATURES)

assert N_FEATURES == 12
assert processed[FEATURE_COLS].notna().all().all()

processed.to_csv(
    os.path.join(EXP_DIR, "processed_dataset.csv"),
    index=False
)


# ============================================================


## 6. Feature Scaling (Training Data Only)

In [ ]:
# 6. SCALING - TRAINING DATA ONLY
# ============================================================

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_X.fit(
    processed.iloc[:train_end][FEATURE_COLS]
)

scaler_y.fit(
    processed.iloc[:train_end][["target_original"]]
    .dropna()
)

X_scaled = scaler_X.transform(
    processed[FEATURE_COLS]
).astype(np.float32)

y_scaled = np.full(
    n_total,
    np.nan,
    dtype=np.float32
)

valid_target = (
    processed["target_original"]
    .notna()
    .to_numpy()
)

y_scaled[valid_target] = (
    scaler_y.transform(
        processed.loc[
            valid_target,
            ["target_original"]
        ]
    ).ravel().astype(np.float32)
)

joblib.dump(
    scaler_X,
    os.path.join(EXP_DIR, "scaler_X.joblib")
)

joblib.dump(
    scaler_y,
    os.path.join(EXP_DIR, "scaler_y.joblib")
)

print("\n" + "=" * 70)
print("SCALING")
print("=" * 70)

print("Scaled X:", X_scaled.shape)
print("Scaled y:", y_scaled.shape)
print("Missing scaled targets:", np.isnan(y_scaled).sum())

assert X_scaled.shape == (8016, 12)
assert y_scaled.shape == (8016,)
assert np.isfinite(X_scaled).all()
assert np.isnan(y_scaled).sum() == 206


# ============================================================


## 7. Generate LSTM Input Sequences

In [ ]:
# 7. GENERATE 24-HOUR INPUT SEQUENCES
# ============================================================

def make_sequences(
    X_data,
    y_data,
    timestamps,
    lookback,
    train_end,
    val_end
):

    containers = {
        split: {
            "X": [],
            "y": [],
            "time": []
        }
        for split in ["train", "val", "test"]
    }

    for i in range(lookback, len(y_data)):

        # Do not predict/evaluate missing actual targets.
        if not np.isfinite(y_data[i]):
            continue

        window = X_data[i-lookback:i]

        if not np.isfinite(window).all():
            continue

        if i < train_end:
            split = "train"
        elif i < val_end:
            split = "val"
        else:
            split = "test"

        containers[split]["X"].append(window)
        containers[split]["y"].append(y_data[i])
        containers[split]["time"].append(timestamps[i])

    result = {}

    for split in ["train", "val", "test"]:

        result[split] = {
            "X": np.asarray(
                containers[split]["X"],
                dtype=np.float32
            ).reshape(-1, lookback, X_data.shape[1]),

            "y": np.asarray(
                containers[split]["y"],
                dtype=np.float32
            ).reshape(-1, 1),

            "time": pd.DatetimeIndex(
                containers[split]["time"]
            )
        }

    return result


data = make_sequences(
    X_scaled,
    y_scaled,
    processed["date_hour"].to_numpy(),
    LOOKBACK,
    train_end,
    val_end
)

X_train = data["train"]["X"]
y_train = data["train"]["y"]
time_train = data["train"]["time"]

X_val = data["val"]["X"]
y_val = data["val"]["y"]
time_val = data["val"]["time"]

X_test = data["test"]["X"]
y_test = data["test"]["y"]
time_test = data["test"]["time"]

print("\n" + "=" * 70)
print("EXPERIMENT K - SEQUENCE SHAPES")
print("=" * 70)

for name, X, y, timestamps in [
    ("Training", X_train, y_train, time_train),
    ("Validation", X_val, y_val, time_val),
    ("Testing", X_test, y_test, time_test)
]:
    print(
        f"{name}: X={X.shape}, y={y.shape} | "
        f"{timestamps.min()} to {timestamps.max()}"
    )

assert X_train.shape == (5403, 24, 12)
assert X_val.shape == (1187, 24, 12)
assert X_test.shape == (1196, 24, 12)


# ============================================================


## 8. Build LSTM Model

In [ ]:
# 8. BUILD LSTM MODEL
# SAME ARCHITECTURE AS EXPERIMENT F
# ============================================================

model = Sequential([
    Input(shape=(LOOKBACK, N_FEATURES)),

    LSTM(64, return_sequences=True),
    Dropout(0.2),

    LSTM(32, return_sequences=True),
    Dropout(0.2),

    LSTM(16),
    Dropout(0.2),

    Dense(1)
])

model.compile(
    optimizer=Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=1.0
    ),
    loss="mse",
    metrics=["mae"]
)

print("\n" + "=" * 70)
print("LSTM MODEL ARCHITECTURE")
print("=" * 70)

model.summary()


# ============================================================


## 9. Train LSTM Model

In [ ]:
# 9. TRAIN MODEL
# ============================================================

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=1
    ),

    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=10,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,
    callbacks=callbacks,
    verbose=1
)

history_df = pd.DataFrame(history.history)

history_df.to_csv(
    os.path.join(EXP_DIR, "training_history.csv"),
    index=False
)

best_epoch = (
    np.argmin(history.history["val_loss"]) + 1
)

print("\n" + "=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)

print("Epochs trained:", len(history_df))
print("Best epoch:", best_epoch)
print(
    "Best validation loss:",
    float(np.min(history.history["val_loss"]))
)


# ============================================================


## 10. Load Best Saved Model

In [ ]:
# 10. LOAD BEST SAVED MODEL
# ============================================================

best_model = tf.keras.models.load_model(
    MODEL_PATH
)


# ============================================================


## 11. Define Evaluation Function

In [ ]:
# 11. EVALUATION FUNCTION
# ============================================================

def evaluate_split(
    model,
    X,
    y,
    timestamps,
    split_name
):

    pred_scaled = model.predict(
        X,
        verbose=0
    )

    pred = scaler_y.inverse_transform(
        pred_scaled.reshape(-1, 1)
    ).ravel()

    actual = scaler_y.inverse_transform(
        y.reshape(-1, 1)
    ).ravel()

    rmse = np.sqrt(
        mean_squared_error(actual, pred)
    )

    mae = mean_absolute_error(actual, pred)

    r2 = r2_score(actual, pred)

    predictions = pd.DataFrame({
        "date_hour": timestamps,
        "actual_kwh": actual,
        "predicted_kwh": pred
    })

    predictions["error_kwh"] = (
        predictions["predicted_kwh"]
        - predictions["actual_kwh"]
    )

    metrics = {
        "Dataset": split_name,
        "Samples": len(actual),
        "RMSE (kWh)": rmse,
        "MAE (kWh)": mae,
        "R² Score": r2
    }

    return metrics, predictions


# ============================================================


## 12. Evaluate Training, Validation and Testing

In [ ]:
# 12. EVALUATE ALL SPLITS
# ============================================================

metrics_train, pred_train = evaluate_split(
    best_model,
    X_train,
    y_train,
    time_train,
    "Training"
)

metrics_val, pred_val = evaluate_split(
    best_model,
    X_val,
    y_val,
    time_val,
    "Validation"
)

metrics_test, pred_test = evaluate_split(
    best_model,
    X_test,
    y_test,
    time_test,
    "Testing"
)

metrics_df = pd.DataFrame([
    metrics_train,
    metrics_val,
    metrics_test
])

print("\n" + "=" * 70)
print("EXPERIMENT K - PERFORMANCE SUMMARY")
print("=" * 70)

print(
    metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================


## 13. Save Evaluation Results

In [ ]:
# 13. SAVE RESULTS
# ============================================================

metrics_df.to_csv(
    os.path.join(EXP_DIR, "all_split_metrics.csv"),
    index=False
)

for split_name, predictions in [
    ("training", pred_train),
    ("validation", pred_val),
    ("test", pred_test)
]:

    predictions.to_csv(
        os.path.join(
            EXP_DIR,
            f"{split_name}_predictions.csv"
        ),
        index=False
    )

print("\nPrediction and metric files saved.")


# ============================================================


## 14. Plot Training and Validation Loss

In [ ]:
# 14. PLOT TRAINING HISTORY
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE (Scaled)")
plt.title("Experiment K - Training History")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(EXP_DIR, "training_loss.png"),
    dpi=300
)

plt.show()


# ============================================================


## 15. Plot Actual vs Predicted Test Demand

In [ ]:
# 15. PLOT TEST PREDICTIONS
# ============================================================

plt.figure(figsize=(14, 5))

plt.plot(
    pred_test["date_hour"],
    pred_test["actual_kwh"],
    label="Actual",
    linewidth=1
)

plt.plot(
    pred_test["date_hour"],
    pred_test["predicted_kwh"],
    label="LSTM K - Historical + Traffic + Weather",
    linewidth=1
)

plt.xlabel("Date")
plt.ylabel("Charging Demand (kWh)")
plt.title("Experiment K - Test Predictions")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        EXP_DIR,
        "test_actual_vs_predicted.png"
    ),
    dpi=300
)

plt.show()


# ============================================================


## 16. Plot First 168 Test Observations

In [ ]:
# 16. PLOT FIRST 168 TEST OBSERVATIONS
# ============================================================

week_df = pred_test.head(168)

plt.figure(figsize=(14, 5))

plt.plot(
    week_df["date_hour"],
    week_df["actual_kwh"],
    label="Actual",
    linewidth=1.5
)

plt.plot(
    week_df["date_hour"],
    week_df["predicted_kwh"],
    label="LSTM K",
    linewidth=1.5
)

plt.xlabel("Date")
plt.ylabel("Charging Demand (kWh)")
plt.title("Experiment K - First 168 Test Observations")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        EXP_DIR,
        "test_first_168_hours.png"
    ),
    dpi=300
)

plt.show()


# ============================================================


## 17. Matched-Timestamp Model Comparison

In [ ]:
# 17. MATCHED-TIMESTAMP COMPARISON
# C, F, H, I, J, K
# ============================================================

comparison_sources = {
    "C - All Features": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_C_2019",
        "test_predictions.csv"
    ),

    "F - Historical Only": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_F_Historical_Only_2019",
        "test_predictions.csv"
    ),

    "H - Historical + Traffic": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_H_Historical_Traffic_2019",
        "test_predictions.csv"
    ),

    "I - Historical + Weather": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_I_Historical_Weather_2019",
        "test_predictions.csv"
    ),

    "J - Historical + EV Sales": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_J_Historical_EV_Sales_2019",
        "test_predictions.csv"
    ),

    "K - Historical + Traffic + Weather": os.path.join(
        EXP_DIR,
        "test_predictions.csv"
    )
}

frames = {}

for model_name, path in comparison_sources.items():

    if not os.path.exists(path):
        print("Skipping missing:", path)
        continue

    temp = pd.read_csv(
        path,
        parse_dates=["date_hour"]
    )

    temp.columns = [
        col.strip().lower()
        for col in temp.columns
    ]

    required = [
        "date_hour",
        "actual_kwh",
        "predicted_kwh"
    ]

    if not set(required).issubset(temp.columns):
        print("Skipping unsupported columns:", path)
        continue

    temp = temp[required].copy()

    if temp["date_hour"].duplicated().any():
        raise ValueError(
            f"Duplicate timestamps in {model_name}"
        )

    frames[model_name] = temp

assert len(frames) >= 2

common_times = None

for frame in frames.values():

    timestamps = set(frame["date_hour"])

    common_times = (
        timestamps
        if common_times is None
        else common_times & timestamps
    )

common_times = sorted(common_times)

if not common_times:
    raise ValueError(
        "No common testing timestamps."
    )

print("\n" + "=" * 70)
print("MATCHED-TIMESTAMP COMPARISON")
print("=" * 70)

print("Models included:", list(frames.keys()))
print("Common samples:", len(common_times))

comparison_results = []
reference_actual = None

for model_name, frame in frames.items():

    matched = (
        frame[
            frame["date_hour"].isin(common_times)
        ]
        .sort_values("date_hour")
        .reset_index(drop=True)
    )

    actual = matched["actual_kwh"].to_numpy()
    predicted = matched["predicted_kwh"].to_numpy()

    if reference_actual is None:
        reference_actual = actual
    else:
        assert np.allclose(
            reference_actual,
            actual,
            rtol=1e-5,
            atol=1e-3
        ), "Actual demand values differ between models."

    comparison_results.append({
        "Model": model_name,
        "Samples": len(actual),
        "RMSE (kWh)": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "MAE (kWh)": mean_absolute_error(
            actual, predicted
        ),
        "R² Score": r2_score(actual, predicted)
    })

comparison_df = pd.DataFrame(
    comparison_results
)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

comparison_df.to_csv(
    os.path.join(
        EXP_DIR,
        "comparison_C_F_H_I_J_K_matched_metrics.csv"
    ),
    index=False
)


# ============================================================


## 18. Save Experiment Configuration

In [ ]:
# 18. TRAFFIC + WEATHER CONTRIBUTION
# ============================================================

if {
    "F - Historical Only",
    "K - Historical + Traffic + Weather"
}.issubset(set(comparison_df["Model"])):

    rmse_f = comparison_df.loc[
        comparison_df["Model"] == "F - Historical Only",
        "RMSE (kWh)"
    ].iloc[0]

    rmse_k = comparison_df.loc[
        comparison_df["Model"] ==
        "K - Historical + Traffic + Weather",
        "RMSE (kWh)"
    ].iloc[0]

    improvement = (
        (rmse_f - rmse_k) / rmse_f
    ) * 100

    print("\n" + "=" * 70)
    print("TRAFFIC + WEATHER CONTRIBUTION")
    print("=" * 70)

    print(
        f"RMSE improvement relative to F: "
        f"{improvement:.2f}%"
    )

    if improvement > 0:
        print("Traffic + weather reduced RMSE.")
    elif improvement < 0:
        print("Traffic + weather increased RMSE.")
    else:
        print("No RMSE difference.")


# ============================================================
# 19. SAVE EXPERIMENT MANIFEST
# ============================================================

manifest = {
    "experiment": "LSTM K",
    "dataset": "Dataset 4 - 2019",
    "features": FEATURE_COLS,
    "n_features": N_FEATURES,
    "lookback": LOOKBACK,
    "forecast_horizon": 1,
    "split": [0.70, 0.15, 0.15],
    "architecture": "LSTM 64-32-16",
    "dropout": 0.2,
    "loss": "mse",
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_epochs": EPOCHS,
    "seed": SEED,
    "test_samples": len(y_test),
    "matched_comparison_samples": len(common_times)
}

with open(
    os.path.join(EXP_DIR, "manifest.json"),
    "w"
) as f:
    json.dump(manifest, f, indent=4)

print("\n" + "=" * 70)
print("EXPERIMENT K COMPLETED SUCCESSFULLY")
print("=" * 70)

print("Best model:", MODEL_PATH)
print("Results directory:", EXP_DIR)
